# Foundation model: Prithvi-EO-2.0

A second, independent look at the change between 2018 and 2023. Instead of a rule written by hand on spectral indices, a foundation model describes each patch of the AOI in both periods, and the distance between the two descriptions is used as a measure of change.

Prithvi-EO-2.0 (IBM and NASA) is a Vision Transformer pretrained on millions of HLS images, the harmonised Landsat and Sentinel-2 product at 30 m, by learning to reconstruct masked parts of each image. It was never trained to detect deforestation: whether its notion of change matches PRODES is exactly what this notebook tests.

This notebook runs in its own environment (`environment-gfm.yml`), since PyTorch and TerraTorch are much heavier than the rest of the project. It reads the band composites written by `01_detection.ipynb` to `data/` and downloads no imagery.

## 1. Settings

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import rioxarray
import torch
from terratorch.registry import BACKBONE_REGISTRY
from terratorch.models.backbones import prithvi_vit

DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
FIGURE_DIR = OUTPUT_DIR / "figures"

MODEL_NAME = "prithvi_eo_v2_300"
GFM_BANDS = ["B02", "B03", "B04", "B8A", "B11", "B12"]

C:\Users\angel\anaconda3\envs\deforestation-gfm\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\angel\anaconda3\envs\deforestation-gfm\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


## 2. The band composites

The per-pixel medians of the six bands, one file per period, on the same 10 m UTM grid as the NDVI change map. Before preparing them for the model, three things are checked: that the grid is the expected one, that the bands are in the right order, and whether any pixel is missing.

In [2]:
baseline = rioxarray.open_rasterio(DATA_DIR / "bands_median_baseline.tif")
recent = rioxarray.open_rasterio(DATA_DIR / "bands_median_recent.tif")

print("shape:", baseline.shape)
print("CRS:", baseline.rio.crs, "| resolution:", baseline.rio.resolution())
print("bands:", baseline.attrs.get("long_name"))
print("missing pixels:", int(baseline.isnull().sum()), "baseline,", int(recent.isnull().sum()), "recent")

band_means = pd.DataFrame({
    "baseline": baseline.mean(dim=("y", "x")).values,
    "recent": recent.mean(dim=("y", "x")).values,
}, index=GFM_BANDS)
band_means.round(4)

shape: (6, 1108, 1100)
CRS: EPSG:32720 | resolution: (10.0, -10.0)
bands: ('B02', 'B03', 'B04', 'B8A', 'B11', 'B12')
missing pixels: 0 baseline, 0 recent


,baseline,recent
B02,0.0531,0.0588
B03,0.0780,0.0842
B04,0.0859,0.0917
B8A,0.3063,0.3068
B11,0.3002,0.3039
B12,0.1780,0.1847


## 3. Loading the model

TerraTorch keeps a registry of pretrained backbones, so the model is built from its name. The 300M version is used, a middle ground between the 100M and 600M variants that still runs on a CPU. The weights, about 1.3 GB, are downloaded from Hugging Face the first time.

`eval()` switches off the behaviour that only matters during training, such as dropout: the model is used here only to describe images, not to learn.

model = BACKBONE_REGISTRY.build(MODEL_NAME, pretrained=True)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"{MODEL_NAME}: {n_params / 1e6:.0f} million parameters")

In [ ]:
model = BACKBONE_REGISTRY.build(MODEL_NAME, pretrained=True)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"{MODEL_NAME}: {n_params / 1e6:.0f} million parameters")

C:\Users\angel\anaconda3\envs\deforestation-gfm\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\angel\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


## 4. The statistics the model expects

During pretraining every band was normalised with a fixed mean and standard deviation, computed on the HLS training data. The same values have to be applied here, otherwise the model receives inputs outside the range it has learned to read. They are shown next to the band means of the composites, to compare the two scales before any preparation.

In [ ]:
model_stats = pd.DataFrame({
    "prithvi_mean": prithvi_vit.PRITHVI_V2_MEAN,
    "prithvi_std": prithvi_vit.PRITHVI_V2_STD,
    "composite_mean_2018": band_means["baseline"].values,
}, index=GFM_BANDS)
model_stats.round(4)

## 5. A dry run

A blank input of the expected size, one image with six bands of 224 × 224 pixels, goes through the model to check the shape of what comes out before real data is used. The image is cut into patches of 16 × 16 pixels, a grid of 14 × 14 = 196 patches. The model returns one output per transformer block, 24 in total, each holding 197 vectors of 1024 numbers: one per patch plus a summary token for the whole image.

In [ ]:
with torch.no_grad():
    dry_run = model(torch.zeros(1, len(GFM_BANDS), 224, 224))

print("outputs (one per block):", len(dry_run))
print("shape of the last one:", tuple(dry_run[-1].shape))